<a href="https://colab.research.google.com/github/sergeyarefjev/wind_predict/blob/main/wind_predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install pytorch==2.10.0+cpu
#!pip install numpy==2.0.2
#!pip install pandas==2.2.2
#!pip install xgboost==3.2.0
#!pip install lightgbm==4.6.0
#!pip install sklearn==1.6.1
!pip install optuna==4.8.0
!pip install catboost==1.2.10
!pip install pytabkit==1.7.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import requests
import time

from statsmodels.tsa.stattools import adfuller
from statsmodels.nonparametric.smoothers_lowess import lowess

from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.model_selection import train_test_split, KFold, TimeSeriesSplit, cross_val_score
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.neighbors import KernelDensity, KNeighborsRegressor

import optuna
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from pytabkit import TabM_D_Regressor

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'optuna'

In [ ]:
import catboost
import sklearn
import pytabkit

print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"XGBoost: {xgb.__version__}")
print(f"LightGBM: {lgb.__version__}")
print(f"CatBoost: {catboost.__version__}")
print(f"sklearn: {sklearn.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"CuDNN: {torch.backends.cudnn.version()}")

import importlib.metadata
version = importlib.metadata.version('pytabkit')
print(f"PyTabkit: {version}")

In [ ]:
def set_seed(seed=42):
    # Python
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # Pandas (через NumPy)
    pd.core.common.random_state(seed)

    # PyTorch (CPU)
    torch.manual_seed(seed)

    # PyTorch (GPU)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # CuDNN (детерминизм)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
SEED = 42
set_seed(SEED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Постпроцессинг

In [ ]:
class ClipRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, regressor, low=0, up=90.09):
        self.regressor = regressor
        self.low = low
        self.up = up

    def fit(self, X, y, sample_weight=None):
        self.regressor.fit(X, y)#, sample_weight=sample_weight)
        return self

    def predict(self, X):
        pred = self.regressor.predict(X)
        return np.clip(pred, a_min=self.low, a_max=self.up)

    def get_params(self, deep=True):
        return {'regressor': self.regressor, 'low': self.low, 'up': self.up}

    def set_params(self, **params):
        for key, value in params.items():
            setattr(self, key, value)
        return self

#ETL

##Загрузка данных

In [ ]:
train = pd.read_csv("/content/drive/MyDrive/ML_data/wind_farm/train_dataset.csv")
train["METEOFORECASTHOUR_OPENM_Datetime"] = pd.to_datetime(
    train["METEOFORECASTHOUR_OPENM_Datetime"])
train.rename(columns={"METEOFORECASTHOUR_OPENM_Datetime": "timestamp",
                      "Кол-во_ВЭУ_в_ремонте": "count_repair",
                      "Выработка. Результирующий расчет": "power"}, inplace=True)
TARGET_COL = "power"
train.sort_values(["timestamp"], ascending=True, inplace=True)
train.reset_index(inplace=True)
train.drop(columns=["index"], inplace=True)
print(train.shape)
train.head()

In [ ]:
valid = pd.read_csv("/content/drive/MyDrive/ML_data/wind_farm/valid_features.csv")
print(valid.shape)
valid["METEOFORECASTHOUR_OPENM_Datetime"] = pd.to_datetime(
    valid["METEOFORECASTHOUR_OPENM_Datetime"])
valid.rename(columns={"METEOFORECASTHOUR_OPENM_Datetime": "timestamp",
                      "Кол-во_ВЭУ_в_ремонте": "count_repair"}, inplace=True)
valid.sort_values(["timestamp"], ascending=True, inplace=True)
valid.reset_index(inplace=True)
valid.drop(columns=["index"], inplace=True)
valid.tail()

###Мета данные

In [ ]:
parsing = False
flag_save = True
if parsing:
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 46.8268455973,
        "longitude": 38.7179393185,
        "start_date": "2022-01-01",
        "end_date": "2026-04-01",
        "hourly": [
            "wind_speed_10m", "wind_speed_80m", "wind_speed_120m", "wind_speed_180m",
            "wind_direction_10m", "wind_direction_80m", "wind_direction_120m", "wind_direction_180m",
            "wind_gusts_10m",
            "temperature_2m", "temperature_80m", "temperature_120m",
            "pressure_msl", "relative_humidity_2m",
            "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
            "rain", "snowfall", "precipitation"
        ] + [f"wind_direction_{h}m" for h in range(0, 105, 5)] + [f"wind_speed_{h}m" for h in range(0, 105, 5)],
        "timezone": "Europe/Moscow"
    }

    response = requests.get(url, params=params)
    data = response.json()
    add_data_df = pd.DataFrame()

    for k, v in data["hourly"].items():
        add_data_df[k] = v

    colna = add_data_df.isna().sum() != 0
    colna = add_data_df.columns[colna]
    add_data_df.drop(columns=colna, inplace=True)
    for col in [col for col in add_data_df.columns if "speed" in col]:
        add_data_df[col] = add_data_df[col] / 3.6
    for col in [col for col in add_data_df.columns if "direction" in col]:
        add_data_df[col] = add_data_df[col] / 1000
    for col in ["relative_humidity_2m", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high"]:
        add_data_df[col] = add_data_df[col] / 100
    add_data_df["time"] = pd.to_datetime(add_data_df["time"])
    add_data_df["month"] = add_data_df["time"].dt.month
    add_data_df["hour_of_day"] = add_data_df["time"].dt.hour
    add_data_df.rename(columns={"time": "timestamp"}, inplace=True)
    add_data_df.head()

    if flag_save:
        add_data_df.to_csv("/content/drive/MyDrive/ML_data/wind_farm/meta_data_from_open_weather.csv")
else:
    add_data_df = pd.read_csv("/content/drive/MyDrive/ML_data/weather_data/meta_data_from_open_weather.csv")
    add_data_df = add_data_df[add_data_df.columns[1:]]
    add_data_df["timestamp"] = pd.to_datetime(add_data_df["timestamp"])

In [ ]:
use_columns_add_data = [col for col in add_data_df.columns if col not in ["month", "hour_of_day"]]
train = train.merge(add_data_df[use_columns_add_data], on="timestamp", how="left",
                    suffixes=("", "_meta"))
valid = valid.merge(add_data_df[use_columns_add_data], on="timestamp", how="left",
                    suffixes=("", "_meta"))

##Загрузка тестовых данных и коррекция загруженных тренировочных данных (отсев колонок)

In [ ]:
test = pd.read_csv("/content/drive/MyDrive/ML_data/wind_farm/test_data.csv")
len_test_cols = len(test.columns)
test["METEOFORECASTHOUR_OPENM_Datetime"] = pd.to_datetime(
    test["METEOFORECASTHOUR_OPENM_Datetime"])
test.rename(columns={"METEOFORECASTHOUR_OPENM_Datetime": "timestamp",
                      "Кол-во_ВЭУ_в_ремонте": "count_repair",
                      "Выработка. Результирующий расчет": "power"}, inplace=True)

test["timestamp"] = pd.to_datetime(test["timestamp"]).dt.round("H")
test.sort_values(["timestamp"], ascending=True, inplace=True)
test.reset_index(inplace=True)
test.drop(columns=["index"], inplace=True)
print(test.shape)
test.tail()

In [ ]:
test.isna().sum() #Пропуски только в power, значит вставим в них нули (чтобы дальше эти строки не удалились)

In [ ]:
test.fillna(0, inplace=True)

### Загрузка мета данных

In [ ]:
parsing = False
flag_save = True
if parsing:
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 46.8268455973,
        "longitude": 38.7179393185,
        "start_date": "2026-04-01",
        "end_date": "2026-05-17",
        "hourly": [
            "wind_speed_10m", "wind_speed_80m", "wind_speed_120m", "wind_speed_180m",
            "wind_direction_10m", "wind_direction_80m", "wind_direction_120m", "wind_direction_180m",
            "wind_gusts_10m",
            "temperature_2m", "temperature_80m", "temperature_120m",
            "pressure_msl", "relative_humidity_2m",
            "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
            "rain", "snowfall", "precipitation"
        ] + [f"wind_direction_{h}m" for h in range(0, 105, 5)] + [f"wind_speed_{h}m" for h in range(0, 105, 5)],
        "timezone": "Europe/Moscow"
    }

    response = requests.get(url, params=params)
    data = response.json()
    add_data_df = pd.DataFrame()

    for k, v in data["hourly"].items():
        add_data_df[k] = v

    colna = add_data_df.isna().sum() != 0
    colna = add_data_df.columns[colna]
    add_data_df.drop(columns=colna, inplace=True)
    for col in [col for col in add_data_df.columns if "speed" in col]:
        add_data_df[col] = add_data_df[col] / 3.6
    for col in [col for col in add_data_df.columns if "direction" in col]:
        add_data_df[col] = add_data_df[col] / 1000
    for col in ["relative_humidity_2m", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high"]:
        add_data_df[col] = add_data_df[col] / 100
    add_data_df["time"] = pd.to_datetime(add_data_df["time"])
    add_data_df["month"] = add_data_df["time"].dt.month
    add_data_df["hour_of_day"] = add_data_df["time"].dt.hour
    add_data_df.rename(columns={"time": "timestamp"}, inplace=True)
    add_data_df.head()

    if flag_save:
        add_data_df.to_csv("/content/drive/MyDrive/ML_data/wind_farm/traintest_meta_data_from_open_weather.csv")
else:
    add_data_df = pd.read_csv("/content/drive/MyDrive/ML_data/wind_farm/traintest_meta_data_from_open_weather.csv")
    add_data_df = add_data_df[add_data_df.columns[1:]]
    add_data_df["timestamp"] = pd.to_datetime(add_data_df["timestamp"])

In [ ]:
use_columns_add_data = [col for col in add_data_df.columns if col not in ["month", "hour_of_day"]]
test_1 = test[:-24].merge(add_data_df[use_columns_add_data], on="timestamp", how="left",
                    suffixes=("", "_meta"))

In [ ]:
parsing = True
flag_save = True
if parsing:
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": 46.8268455973,
        "longitude": 38.7179393185,
        "hourly": [
            "wind_speed_10m", "wind_speed_80m", "wind_speed_120m", "wind_speed_180m",
            "wind_direction_10m", "wind_direction_80m", "wind_direction_120m", "wind_direction_180m",
            "wind_gusts_10m",
            "temperature_2m", "temperature_80m", "temperature_120m",
            "pressure_msl", "relative_humidity_2m",
            "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
            "rain", "snowfall", "precipitation"
        ] + [f"wind_direction_{h}m" for h in range(0, 105, 5)] + [f"wind_speed_{h}m" for h in range(0, 105, 5)],
            "timezone": "Europe/Moscow",
            "start_date": "2026-05-18",
            "end_date": "2026-05-18",
            "models": "icon_seamless"
    }

    response = requests.get(url, params=params)
    data = response.json()
    add_data_df = pd.DataFrame()

    for k, v in data["hourly"].items():
        add_data_df[k] = v

    colna = add_data_df.isna().sum() != 0
    colna = add_data_df.columns[colna]
    add_data_df.drop(columns=colna, inplace=True)
    for col in [col for col in add_data_df.columns if "speed" in col]:
        add_data_df[col] = add_data_df[col] / 3.6
    for col in [col for col in add_data_df.columns if "direction" in col]:
        add_data_df[col] = add_data_df[col] / 1000
    for col in ["relative_humidity_2m", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high"]:
        add_data_df[col] = add_data_df[col] / 100
    add_data_df["time"] = pd.to_datetime(add_data_df["time"])
    add_data_df["month"] = add_data_df["time"].dt.month
    add_data_df["hour_of_day"] = add_data_df["time"].dt.hour
    add_data_df.rename(columns={"time": "timestamp"}, inplace=True)
    add_data_df.head()

    if flag_save:
        add_data_df.to_csv("/content/drive/MyDrive/ML_data/wind_farm/forecast_meta_data_from_open_weather.csv")
else:
    add_data_df = pd.read_csv("/content/drive/MyDrive/ML_data/wind_farm/forecast_meta_data_from_open_weather.csv")
    add_data_df = add_data_df[add_data_df.columns[1:]]
    add_data_df["timestamp"] = pd.to_datetime(add_data_df["timestamp"])

In [ ]:
use_columns_add_data = [col for col in add_data_df.columns if col not in ["month", "hour_of_day"]]
test_2 = test[-24:].merge(add_data_df[use_columns_add_data], on="timestamp", how="left",
                    suffixes=("", "_meta"))

In [ ]:
use_columns = [col for col in test_2.columns if col in test_1.columns]
test = pd.concat([test_1[use_columns], test_2[use_columns]], axis=0)
train = train[test.columns]
valid = valid[[col for col in test.columns if col != TARGET_COL]]

##Детекция и устранение аномалий

In [ ]:

drop_start = True
if drop_start:
    train.dropna(inplace=True)

features = train.columns[4:]

iso = IsolationForest(
    contamination=0.02,
    random_state=SEED,
    n_estimators=500
    )
imputer = KNNImputer(
        n_neighbors=5,
        weights="distance"
    )

flag = iso.fit_predict(train[features]) == -1
train.loc[flag, features] = np.nan
#train[features] = imputer.fit_transform(train[features])
train.dropna(inplace=True)

In [ ]:
"""
drop_start = True
if drop_start:
    train.dropna(inplace=True)

features = train.columns[4:]

iso = IsolationForest(
    contamination=0.02,
    random_state=SEED,
    n_estimators=500
    )
imputer = KNNImputer(
        n_neighbors=5,
        weights="distance"
    )
flag = iso.fit_predict(train[features]) == -1

each_column = False
if each_column:
    for col in features:
        flag = iso.fit_predict(train[col].values.reshape((-1, 1))) == -1
        train.loc[flag, col] = np.nan
        train[col] = imputer.fit_transform(train[col].values.reshape((-1, 1)))
else:
    flag = iso.fit_predict(train[features]) == -1
    train[flag] = np.nan
    train[features] = imputer.fit_transform(train[features])
    """

In [ ]:
train.isna().sum()

##Feature Enginiring

In [ ]:
use_meta = True
use_test = True

In [ ]:
def add_density(data):
    if use_meta:
        data["density_2m"] = data["pressure_msl_meta"] / (data["temperature_2m"] + 273.15)
    data["density_80m"] = data["pressure_msl"] / (data["temperature_80m"] + 273.15)
    data["density_120m"] = data["pressure_msl"] / (data["temperature_120m"] + 273.15)
    print("Плотность добавлена!")
    return True

add_density(train)
add_density(valid)
if use_test:
    add_density(test)

In [ ]:
def add_humidity(data):
    data["humidity"] = data["cloud_cover_low"].copy()
    cols = ["showers", "rain", "snowfall"]
    weights = [0.5, 0.2, 0.3]
    for col, w in zip(cols, weights):
        if data[col].max() > data[col].min():
            data["humidity"] += w * ((data[col] - data[col].min()) /
                                    (data[col].max() - data[col].min()))
    data["humidity"] = ((data["humidity"] - data["humidity"].min()) /
                        (data["humidity"].max() - data["humidity"].min()))
    print("Оценка влажности добавлена!")
    return True

add_humidity(train)
add_humidity(valid)
if use_test:
    add_humidity(test)

In [ ]:
def add_sin_cos(data, columns):
    for col in columns:
        data[f"sin_direction_{col[col.rfind('_') + 1:]}"] = np.sin(
                                                data[col] * 1000 * np.pi / 180)
        data[f"cos_direction_{col[col.rfind('_') + 1:]}"] = np.cos(
                                                data[col] * 1000 * np.pi / 180)

    print("Синус-косинус кодирование добавлено!")
    return True

direction_cols = [col for col in train.columns if "wind_direction" in col and "lag" not in col]
print(direction_cols)

add_sin_cos(train, direction_cols)
add_sin_cos(valid, direction_cols)
if use_test:
    add_sin_cos(test, direction_cols)

In [ ]:
def add_wind_curve(cols):
    for col in cols:
        train[f"{col}_cut"] = train[col].apply(lambda x: round(x))
        valid[f"{col}_cut"] = valid[col].apply(lambda x: round(x))
        if use_test:
            test[f"{col}_cut"] = test[col].apply(lambda x: round(x))
        wind_curve = train.groupby(f"{col}_cut").median()[TARGET_COL]
        max_col = np.max(wind_curve.index)
        for idx in range(max_col, 51):
            wind_curve[idx] = wind_curve[max_col]
        train[f"{col}_curve"] = train[f"{col}_cut"].apply(lambda x: wind_curve[x])
        valid[f"{col}_curve"] = valid[f"{col}_cut"].apply(lambda x: wind_curve[x])
        train.drop(columns=[f"{col}_cut"], inplace=True)
        valid.drop(columns=[f"{col}_cut"], inplace=True)
        if use_test:
            test[f"{col}_cut"] = test[col].apply(lambda x: round(x))
            test[f"{col}_curve"] = test[f"{col}_cut"].apply(lambda x: wind_curve[x])
            test.drop(columns=[f"{col}_cut"], inplace=True)
    print(f"Характерестическая кривая для колонок: {cols} добавлена!")
    return True

add_wind_curve(["wind_speed_80m", "wind_speed_10m_meta"])

In [ ]:
def add_effect_speed(data, columns, eps=1e-3):
    TI = (data["wind_gusts_10m"] - data["wind_speed_10m"]) / (data["wind_speed_10m"] + eps)
    if use_meta:
        TI_add = (data["wind_gusts_10m_meta"] - data["wind_speed_10m_meta"]) / (data["wind_speed_10m_meta"] + eps)
        TI = (TI + TI_add) / 2
    for col in columns:
        data[f"effect_{col}"] = data[col] * (1 - 0.5 * TI**2)
    print("Добавлена эффективная скорость ветра (с поправкой на турбулентность)!")
    return True

effect_columns = [col for col in train.columns if "speed" in col and "degree" not in col and "mean" not in col and "lag" not in col and "effect" not in col]

print(effect_columns)
add_effect_speed(train, effect_columns)
add_effect_speed(valid, effect_columns)
if use_test:
    add_effect_speed(test, effect_columns)

In [ ]:
def add_mean_wind(columns):
    for col in columns:
        add_data = pd.concat([train, valid], axis=0)
        add_data[f"mean_{col}"] = (add_data[col] + add_data[col].shift(1)) / 2
        train[f"mean_{col}"] = add_data[f"mean_{col}"].iloc[:len(train)]
        valid[f"mean_{col}"] = add_data[f"mean_{col}"].iloc[len(train):]
        if use_test:
            test[f"mean_{col}"] = (test[col] + test[col].shift(1)) / 2
    print("Средняя скорость за прошлый и нынешний час добавлена!")
    return True

speed_cols = [col for col in train.columns if "speed" in col and "degree" not in col and "mean" not in col and "lag" not in col]
print(speed_cols)

add_mean_wind(speed_cols)

In [ ]:
def add_degree(data, columns):
    for col in columns:
        data[f"{col}_degree_2"] = data[col]**2
        data[f"{col}_degree_3"] = data[col]**3

    print("Степени добавлены!")
    return True

speed_cols = [col for col in train.columns if "speed" in col and "degree" not in col and "lag" not in col]
print(speed_cols)

add_degree(train, speed_cols)
add_degree(valid, speed_cols)
if use_test:
    add_degree(test, speed_cols)

In [ ]:
def add_kinetic_energy(data):
    for h in [80, 120]:
        data[f"kinetic_energy_{h}m"] = data[f"density_{h}m"] * data[f"wind_speed_{h}m_degree_3"]
        data[f"kinetic_energy_{h}m_mean"] = data[f"density_{h}m"] * data[f"mean_wind_speed_{h}m_degree_3"]
    if use_meta:
        for h in [80, 120]:
            data[f"kinetic_energy_{h}m_from_2m"] = data["density_2m"] * data[f"wind_speed_{h}m_degree_3"]
            data[f"kinetic_energy_{h}m_mean_from_2m"] = data["density_2m"] * data[f"wind_speed_{h}m_degree_3"]
    print("Кинетическая энергия добавлена!")
    return True

add_kinetic_energy(train)
add_kinetic_energy(valid)
if use_test:
    add_kinetic_energy(test)

In [ ]:
def add_projection_enegry(data):
    for h in [80, 120]:
        data[f"sin_projection_energy_{h}m"] = (data[f"kinetic_energy_{h}m"] *
                                           data[f"sin_direction_{h}m"])
        data[f"cos_projection_energy_{h}m"] = (data[f"kinetic_energy_{h}m"] *
                                               data[f"cos_direction_{h}m"])
        if use_meta:
            data[f"sin_projection_energy_{h}m"] = (data[f"kinetic_energy_{h}m_from_2m"] *
                                            data[f"sin_direction_{h}m"])
            data[f"cos_projection_energy_{h}m"] = (data[f"kinetic_energy_{h}m_from_2m"] *
                                                data[f"cos_direction_{h}m"])
    print("Проекция энергии добавлена!") #используется оба кодирования, так как
    #ВЭС'ы стоят под разными углами
    return True

add_projection_enegry(train)
add_projection_enegry(valid)
if use_test:
    add_projection_enegry(test)

In [ ]:
def add_toggle(data, low=5, up=9):
    data["toggle"] = np.zeros(len(data))
    data["toggle"] = np.where(data["wind_speed_80m"] < low, 1, data["toggle"])
    data["toggle"] = np.where(data["wind_speed_80m"] > up, 3, data["toggle"])
    data["toggle"] = np.where((data["wind_speed_80m"] >= low) &
                                (data["wind_speed_80m"] <= up), 2, data["toggle"])
    print("Переключение добавлено!")
    return True



#Эти границы из документации
low = 3
up = 11
add_toggle(train, low=low, up=up)
add_toggle(valid, low=low, up=up)
if use_test:
    add_toggle(test, low=low, up=up)

In [ ]:
def add_hour_seasonal(data):
    data["sin_hour"] = np.sin(data["hour_of_day"] / 24 * 2 * np.pi)
    data["cos_hour"] = np.cos(data["hour_of_day"] / 24 * 2 * np.pi)
    print("Синус-косинус преобразование для часов добавлено!")
    return True

add_hour_seasonal(train)
add_hour_seasonal(valid)
if use_test:
    add_hour_seasonal(test)

In [ ]:
def add_month_seasonal(data):
    data["sin_month"] = np.sin(data["month"] / 12 * 2 * np.pi)
    data["cos_month"] = np.cos(data["month"] / 12 * 2 * np.pi)
    print("Синус-косинус кодирование для часов добавлено!")
    return True

add_month_seasonal(train)
add_month_seasonal(valid)
if use_test:
    add_month_seasonal(test)

In [ ]:
def add_shift_wind(data, eps=1e-5):
    data["shift_wind_10_80"] = np.log((data["wind_speed_80m"] + eps) / (data["wind_speed_10m"] + eps))
    data["shift_wind_80_120"] = np.log((data["wind_speed_120m"] + eps) / (data["wind_speed_80m"] + eps))
    data["shift_wind_120_180"] = np.log((data["wind_speed_180m"] + eps) / (data["wind_speed_120m"] + eps))
    print("Сдвиги ветра добавлены!")
    return True

test_flag=False
if test_flag:
    add_shift_wind(train)
    add_shift_wind(valid)

In [ ]:
def add_sample_weights_from_valid(train_data_orig, test_data_orig):
    scaler = StandardScaler()
    train_data = scaler.fit_transform(train_data_orig)
    test_data = scaler.fit_transform(test_data_orig)
    center = np.mean(test_data, axis=0)

    dist = np.zeros(len(train_data))
    for i in range(train_data.shape[0]):
        dist[i] = np.sum(train_data[i] - center)
    dist_max = np.max(dist)
    dist_inverse = dist_max - dist
    train_weights = dist_inverse
    test_weights = 1
    return train_weights, test_weights

choice = False
if choice:
    all_features_cols = [col for col in train.columns if col not in [
    "timestamp", TARGET_COL
    ]]
    train.dropna(inplace=True)
    train["weights"], valid["weights"] = add_sample_weights_from_valid(train[all_features_cols], valid[all_features_cols])

In [ ]:
def add_time_encoding(train, val):
    time_to_int = np.arange(len(train) + len(val))
    sin_encoding = np.sin(2 * np.pi * time_to_int / len(time_to_int))
    cos_encoding = np.cos(2 * np.pi * time_to_int / len(time_to_int))
    train["sin_time_encoding"] = sin_encoding[:len(train)]
    train["cos_time_encoding"] = cos_encoding[:len(train)]
    val["sin_time_encoding"] = sin_encoding[len(train):]
    val["cos_time_encoding"] = cos_encoding[len(train):]
    print("Синус-косинус кодирование времени добавлено")
    return True

#add_time_encoding(train, valid)

In [ ]:
def add_lags(lags, columns):
    all_data = pd.concat([train, valid], axis=0)
    for col in columns:
        for lag in lags:
            all_data[f"{col}_{lag}"] = all_data[col].shift(lag)
            train[f"{col}_lag_{lag}"] = all_data[f"{col}_{lag}"].iloc[:len(train)]
            valid[f"{col}_lag_{lag}"] = all_data[f"{col}_{lag}"].iloc[len(train):]
            if use_test:
                test[f"{col}_lag_{lag}"] = test[col].shift(lag)
    print(f"Лаги для колонок: {columns} добавлены!")
    return True

lags_columns = [
    "kinetic_energy_80m",
    "kinetic_energy_120m",
    "pressure_msl",
    "temperature_80m",
    "wind_speed_10m_meta",
    "wind_direction_10m_meta",
    "pressure_msl_meta"
]
add_lags([1, 2, 24], lags_columns)

In [ ]:
def add_sqrt_pressure(p_cols):
    """
    Введение этой фичи обусловленно эмпирическим правилом: V ~ sqrt(p)
    Оно работает лишь при сильных изменениях давления и дает оценку для
    порывов ветра
    """
    add_data = pd.concat([train, valid], axis=0)
    for p_col in p_cols:
        add_data[f"sqrt_diff_{p_col}"] = np.where(
            add_data[p_col] - add_data[p_col].shift(1) > 0,
            np.sqrt(add_data[p_col] - add_data[p_col].shift(1)),
        -np.sqrt(add_data[p_col].shift(1) - add_data[p_col]))
        train[f"sqrt_diff_{p_col}"] = add_data[f"sqrt_diff_{p_col}"].iloc[:len(train)]
        valid[f"sqrt_diff_{p_col}"] = add_data[f"sqrt_diff_{p_col}"].iloc[len(train)]

        if use_test:
            test[f"sqrt_diff_{p_col}"] = np.where(
            test[p_col] - test[p_col].shift(1) > 0,
            np.sqrt(test[p_col] - test[p_col].shift(1)),
        -np.sqrt(test[p_col].shift(1) - test[p_col]))
    print("Корень из разности давлений добавлен!")
    return True

add_sqrt_pressure(["pressure_msl", "pressure_msl_meta"])

In [ ]:
small_features_cols = ["month", "hour_of_day", "count_repair", "wind_speed_10m",
                       "wind_speed_80m", "wind_speed_120m",
                       "sin_direction_10m", "sin_direction_80m",
                       "sin_direction_120m","cos_direction_10m",
                       "cos_direction_80m", "cos_direction_120m"]

small_features_and_degree_cols = small_features_cols + [
    col for col in train.columns if "degree" in col
]

lasso_01_features = ['month', 'hour_of_day', 'wind_speed_10m', 'wind_speed_80m',
    'wind_speed_120m','wind_speed_180m', 'wind_gusts_10m', 'temperature_80m',
    'pressure_msl', 'count_repair', 'sin_direction_10m', 'sin_direction_80m',
    'cos_direction_80m', 'sin_direction_120m', 'sin_direction_180m',
    'wind_speed_10m_degree_2', 'wind_speed_10m_degree_3', 'wind_speed_80m_degree_2',
    'wind_speed_80m_degree_3', 'wind_speed_120m_degree_2', 'wind_speed_120m_degree_3',
    'wind_speed_180m_degree_2', 'wind_speed_180m_degree_3']


lasso_01_features = ['month',
 'hour_of_day',
 'wind_speed_10m',
 'wind_speed_80m',
 'wind_speed_120m',
 'wind_speed_180m',
 'wind_gusts_10m',
 'temperature_80m',
 'pressure_msl',
 'count_repair',
 'sin_direction_10m',
 'sin_direction_80m',
 'cos_direction_80m',
 'sin_direction_120m',
 'sin_direction_180m',
 'wind_speed_10m_degree_2',
 'wind_speed_10m_degree_3',
 'wind_speed_80m_degree_2',
 'wind_speed_80m_degree_3',
 'wind_speed_120m_degree_2',
 'wind_speed_120m_degree_3',
 'wind_speed_180m_degree_2',
 'wind_speed_180m_degree_3',
 'sin_hour',
 'cos_hour',
 'sin_month',
 'cos_month']

lasso_03_features = ['month',
 'hour_of_day',
 'wind_speed_120m',
 'temperature_80m',
 'pressure_msl',
 'count_repair',
 'sin_direction_10m',
 'cos_direction_80m',
 'cos_direction_120m',
 'wind_speed_10m_degree_3',
 'wind_speed_80m_degree_2',
 'wind_speed_80m_degree_3',
 'wind_speed_120m_degree_2',
 'wind_speed_120m_degree_3',
 'wind_speed_180m_degree_2',
 'wind_speed_180m_degree_3']

lasso_0007_features = ['month',
 'hour_of_day',
 'wind_speed_10m',
 'wind_speed_80m',
 'wind_speed_120m',
 'wind_speed_180m',
 'wind_direction_10m',
 'wind_direction_120m',
 'wind_direction_180m',
 'wind_gusts_10m',
 'temperature_80m',
 'temperature_120m',
 'pressure_msl',
 'rain',
 'showers',
 'snowfall',
 'cloud_cover_low',
 'count_repair',
 'sin_direction_10m',
 'cos_direction_10m',
 'sin_direction_80m',
 'cos_direction_80m',
 'sin_direction_120m',
 'cos_direction_120m',
 'sin_direction_180m',
 'cos_direction_180m',
 'wind_speed_10m_degree_2',
 'wind_speed_10m_degree_3',
 'wind_speed_80m_degree_2',
 'wind_speed_80m_degree_3',
 'wind_speed_120m_degree_2',
 'wind_speed_120m_degree_3',
 'wind_speed_180m_degree_2',
 'wind_speed_180m_degree_3',
 'toggle',
 'sin_hour',
 'cos_hour',
 'sin_month',
 'cos_month',
 'shift_wind_10_80',
 'shift_wind_80_120',
 'shift_wind_120_180']

hill_climb_from_random = ['month',
 'hour_of_day',
 'wind_speed_10m',
 'wind_speed_80m',
 'wind_speed_180m',
 'wind_direction_10m',
 'wind_gusts_10m',
 'temperature_120m',
 'rain',
 'showers',
 'cloud_cover_low',
 'count_repair',
 'humidity',
 'sin_direction_120m',
 'mean_wind_speed_180m',
 'wind_speed_10m_degree_2',
 'wind_speed_10m_degree_3',
 'mean_wind_speed_10m_degree_3',
 'mean_wind_speed_120m_degree_2',
 'kinetic_energy_80m',
 'kinetic_energy_120m',
 'kinetic_energy_120m_mean',
 'cos_hour',
 'cos_month']

hill_climb_from_random_meta = ['hour_of_day',
 'wind_speed_80m',
 'wind_speed_180m',
 'wind_direction_80m',
 'wind_gusts_10m',
 'temperature_80m',
 'temperature_120m',
 'rain',
 'cloud_cover_low',
 'count_repair',
 'wind_gusts_10m_meta',
 'cloud_cover_mid',
 'rain_meta',
 'snowfall_meta',
 'density_120m',
 'sin_direction_10m',
 'sin_direction_120m',
 'cos_direction_120m',
# 'cos_direction_100m',
# 'effect_wind_speed_100m',
 'mean_wind_speed_80m',
 'mean_wind_speed_120m',
 #'mean_effect_wind_speed_100m',
 'wind_speed_10m_degree_3',
 'wind_speed_180m_degree_3',
 'wind_speed_10m_meta_degree_2',
 #'wind_speed_100m_degree_3',
 'effect_wind_speed_10m_degree_2',
 'effect_wind_speed_10m_degree_3',
 'effect_wind_speed_120m_degree_2',
 'effect_wind_speed_180m_degree_2',
 'effect_wind_speed_180m_degree_3',
 'effect_wind_speed_10m_meta_degree_3',
 #'effect_wind_speed_100m_degree_2',
 'mean_wind_speed_10m_degree_2',
 'mean_wind_speed_80m_degree_2',
 'mean_wind_speed_120m_degree_2',
 'mean_wind_speed_180m_degree_3',
 #'mean_wind_speed_100m_degree_2',
 #'mean_wind_speed_100m_degree_3',
 'mean_effect_wind_speed_180m_degree_2',
 'mean_effect_wind_speed_180m_degree_3',
 'mean_effect_wind_speed_10m_meta_degree_2',
 'kinetic_energy_120m',
 'kinetic_energy_120m_from_2m',
 'sin_projection_energy_120m',
 'toggle',
 'sin_hour',
 'cos_month',
 'kinetic_energy_80m_lag_24',
 'kinetic_energy_120m_lag_1',
 'kinetic_energy_120m_lag_2',
 'kinetic_energy_120m_lag_24',
 'pressure_msl_lag_24',
 'temperature_80m_lag_24']

hill_climb_from_lasso = ['wind_speed_10m',
 'wind_speed_120m',
 'wind_speed_180m',
 'wind_direction_10m',
 'wind_gusts_10m',
 'temperature_80m',
 'showers',
 'cloud_cover_low',
 'count_repair',
 'sin_direction_80m',
 'cos_direction_80m',
 'sin_direction_180m',
 'wind_speed_10m_degree_2',
 'wind_speed_10m_degree_3',
 'wind_speed_80m_degree_2',
 'wind_speed_80m_degree_3',
 'wind_speed_120m_degree_2',
 'wind_speed_120m_degree_3',
 'wind_speed_180m_degree_2',
 'wind_speed_180m_degree_3',
 'toggle',
 'sin_hour',
 'cos_hour',
 'sin_month',
 'cos_month',
 'shift_wind_10_80']

hill_climb_from_all = ['hour_of_day',
 'wind_speed_80m',
 'wind_speed_120m',
 'wind_direction_10m',
 'wind_gusts_10m',
 'temperature_120m',
 'rain',
 'showers',
 'snowfall',
 'cloud_cover_low',
 'count_repair',
 'sin_direction_10m',
 'cos_direction_10m',
 'cos_direction_80m',
 'sin_direction_120m',
 'cos_direction_120m',
 'sin_direction_180m',
 'wind_speed_10m_degree_2',
 'wind_speed_10m_degree_3',
 'wind_speed_80m_degree_2',
 'wind_speed_80m_degree_3',
 'wind_speed_120m_degree_2',
 'wind_speed_120m_degree_3',
 'wind_speed_180m_degree_2',
 'wind_speed_180m_degree_3',
 'toggle',
 'sin_hour',
 'cos_hour',
 'sin_month',
 'cos_month',
 'shift_wind_10_80',
 'shift_wind_80_120',
 'shift_wind_120_180']

hill_climb_from_all_meta=['month',
 'wind_speed_10m',
 'wind_speed_80m',
 'wind_speed_120m',
 'wind_speed_180m',
 'wind_direction_10m',
 'wind_direction_80m',
 'wind_direction_120m',
 'wind_direction_180m',
 'wind_gusts_10m',
 'temperature_80m',
 'temperature_120m',
 'pressure_msl',
 'showers',
 'snowfall',
 'cloud_cover_low',
 'count_repair',
 'wind_speed_10m_meta',
 'wind_direction_10m_meta',
 'wind_gusts_10m_meta',
 'temperature_2m',
 'pressure_msl_meta',
 'relative_humidity_2m',
 'cloud_cover_low_meta',
 'cloud_cover_mid',
 'cloud_cover_high',
 'rain_meta',
 'snowfall_meta',
 'precipitation',
 'wind_direction_100m',
 'wind_speed_100m',
 #'month_meta',
# 'hour_of_day_meta',
 'density_2m',
 'density_80m',
 'density_120m',
 'humidity',
 'sin_direction_10m',
 'cos_direction_10m',
 'sin_direction_80m',
 'cos_direction_80m',
 'sin_direction_120m',
 'cos_direction_120m',
 'sin_direction_180m',
 'cos_direction_180m',
 'sin_direction_meta',
 'cos_direction_meta',
 'sin_direction_100m',
 'cos_direction_100m',
 'effect_wind_speed_10m',
 'effect_wind_speed_80m',
 'effect_wind_speed_120m',
 'effect_wind_speed_180m',
 'effect_wind_speed_10m_meta',
 'effect_wind_speed_100m',
 'mean_wind_speed_10m',
 'mean_wind_speed_80m',
 'mean_wind_speed_120m',
 'mean_wind_speed_180m',
 'mean_wind_speed_10m_meta',
 'mean_wind_speed_100m',
 'mean_effect_wind_speed_10m',
 'mean_effect_wind_speed_80m',
 'mean_effect_wind_speed_120m',
 'mean_effect_wind_speed_180m',
 'mean_effect_wind_speed_10m_meta',
 'mean_effect_wind_speed_100m',
 'wind_speed_10m_degree_2',
 'wind_speed_10m_degree_3',
 'wind_speed_80m_degree_2',
 'wind_speed_80m_degree_3',
 'wind_speed_120m_degree_2',
 'wind_speed_120m_degree_3',
 'wind_speed_180m_degree_2',
 'wind_speed_180m_degree_3',
 'wind_speed_10m_meta_degree_2',
 'wind_speed_10m_meta_degree_3',
 'wind_speed_100m_degree_2',
 'wind_speed_100m_degree_3',
 'effect_wind_speed_10m_degree_2',
 'effect_wind_speed_10m_degree_3',
 'effect_wind_speed_80m_degree_2',
 'effect_wind_speed_80m_degree_3',
 'effect_wind_speed_120m_degree_2',
 'effect_wind_speed_120m_degree_3',
 'effect_wind_speed_180m_degree_2',
 'effect_wind_speed_180m_degree_3',
 'effect_wind_speed_10m_meta_degree_2',
 'effect_wind_speed_10m_meta_degree_3',
 'effect_wind_speed_100m_degree_2',
 'effect_wind_speed_100m_degree_3',
 'mean_wind_speed_10m_degree_2',
 'mean_wind_speed_10m_degree_3',
 'mean_wind_speed_80m_degree_2',
 'mean_wind_speed_80m_degree_3',
 'mean_wind_speed_120m_degree_2',
 'mean_wind_speed_120m_degree_3',
 'mean_wind_speed_180m_degree_2',
 'mean_wind_speed_180m_degree_3',
 'mean_wind_speed_10m_meta_degree_2',
 'mean_wind_speed_10m_meta_degree_3',
 'mean_wind_speed_100m_degree_2',
 'mean_wind_speed_100m_degree_3',
 'mean_effect_wind_speed_10m_degree_2',
 'mean_effect_wind_speed_10m_degree_3',
 'mean_effect_wind_speed_80m_degree_2',
 'mean_effect_wind_speed_80m_degree_3',
 'mean_effect_wind_speed_120m_degree_2',
 'mean_effect_wind_speed_120m_degree_3',
 'mean_effect_wind_speed_180m_degree_2',
 'mean_effect_wind_speed_180m_degree_3',
 'mean_effect_wind_speed_10m_meta_degree_2',
 'mean_effect_wind_speed_10m_meta_degree_3',
 'mean_effect_wind_speed_100m_degree_2',
 'mean_effect_wind_speed_100m_degree_3',
 'kinetic_energy_80m',
 'kinetic_energy_80m_mean',
 'kinetic_energy_120m',
 'kinetic_energy_120m_mean',
 'kinetic_energy_80m_from_2m',
 'kinetic_energy_80m_mean_from_2m',
 'kinetic_energy_120m_from_2m',
 'kinetic_energy_120m_mean_from_2m',
 'sin_projection_energy_80m',
 'cos_projection_energy_80m',
 'sin_projection_energy_120m',
 'cos_projection_energy_120m',
 'toggle',
 'sin_hour',
 'cos_hour',
 'sin_month',
 'cos_month',
 'kinetic_energy_80m_lag_1',
 'kinetic_energy_80m_lag_2',
 'kinetic_energy_80m_lag_24',
 'kinetic_energy_120m_lag_1',
 'kinetic_energy_120m_lag_2',
 'kinetic_energy_120m_lag_24',
 'pressure_msl_lag_1',
 'pressure_msl_lag_2',
 'pressure_msl_lag_24',
 'temperature_80m_lag_1',
 'temperature_80m_lag_2',
 'temperature_80m_lag_24',
 'wind_speed_10m_meta_lag_1',
 'wind_speed_10m_meta_lag_2',
 'wind_speed_10m_meta_lag_24',
 'wind_direction_10m_meta_lag_1',
 'wind_direction_10m_meta_lag_2',
 'wind_direction_10m_meta_lag_24',
 'sqrt_diff_pressure_msl',
 'sqrt_diff_pressure_msl_meta']

hill_climb_from_meta_meta = ['wind_direction_10m',
 'wind_direction_80m',
 'wind_direction_120m',
 'wind_gusts_10m',
 'temperature_120m',
 'pressure_msl',
 'rain',
 'showers',
 'snowfall',
 'cloud_cover_low',
 'count_repair',
 'wind_speed_10m_meta',
 'wind_direction_10m_meta',
 'pressure_msl_meta',
 'cloud_cover_low_meta',
 'cloud_cover_mid',
 'cloud_cover_high',
 'rain_meta',
 'snowfall_meta',
 'precipitation',
 'wind_direction_100m',
 'wind_speed_100m',
 'density_2m',
 'density_80m',
 'density_120m',
 'humidity',
 'sin_direction_10m',
 'sin_direction_80m',
 'cos_direction_80m',
 'sin_direction_120m',
 'cos_direction_120m',
 'sin_direction_180m',
 'cos_direction_180m',
 'sin_direction_meta',
 'cos_direction_meta',
 'sin_direction_100m',
 'cos_direction_100m',
 #'wind_speed_80m_cut',
 'wind_speed_80m_curve',
 'effect_wind_speed_10m',
 'effect_wind_speed_80m',
 'effect_wind_speed_120m',
 'effect_wind_speed_180m',
 'effect_wind_speed_10m_meta',
 'effect_wind_speed_100m',
 #'effect_wind_speed_80m_cut',
 'effect_wind_speed_80m_curve',
 'mean_wind_speed_10m',
 'mean_wind_speed_80m',
 'mean_wind_speed_120m',
 'mean_wind_speed_180m',
 'mean_wind_speed_10m_meta',
 'mean_wind_speed_100m',
# 'mean_wind_speed_80m_cut',
 'mean_wind_speed_80m_curve',
 'mean_effect_wind_speed_10m',
 'mean_effect_wind_speed_80m',
 'mean_effect_wind_speed_120m',
 'mean_effect_wind_speed_180m',
 'mean_effect_wind_speed_10m_meta',
 'mean_effect_wind_speed_100m',
 #'mean_effect_wind_speed_80m_cut',
 'mean_effect_wind_speed_80m_curve',
 'wind_speed_10m_degree_2',
 'wind_speed_10m_degree_3',
 'wind_speed_80m_degree_2',
 'wind_speed_80m_degree_3',
 'wind_speed_120m_degree_2',
 'wind_speed_120m_degree_3',
 'wind_speed_180m_degree_2',
 'wind_speed_180m_degree_3',
 'wind_speed_10m_meta_degree_2',
 'wind_speed_10m_meta_degree_3',
 'wind_speed_100m_degree_2',
 'wind_speed_100m_degree_3',
 #'wind_speed_80m_cut_degree_2',
 #'wind_speed_80m_cut_degree_3',
 'wind_speed_80m_curve_degree_2',
 'wind_speed_80m_curve_degree_3',
 'effect_wind_speed_10m_degree_2',
 'effect_wind_speed_10m_degree_3',
 'effect_wind_speed_80m_degree_2',
 'effect_wind_speed_80m_degree_3',
 'effect_wind_speed_120m_degree_3',
 'effect_wind_speed_180m_degree_2',
 'effect_wind_speed_180m_degree_3',
 'effect_wind_speed_10m_meta_degree_2',
 'effect_wind_speed_10m_meta_degree_3',
 'effect_wind_speed_100m_degree_2',
 'effect_wind_speed_100m_degree_3',
# 'effect_wind_speed_80m_cut_degree_2',
# 'effect_wind_speed_80m_cut_degree_3',
 'effect_wind_speed_80m_curve_degree_2',
 'effect_wind_speed_80m_curve_degree_3',
 'mean_wind_speed_10m_degree_2',
 'mean_wind_speed_10m_degree_3',
 'mean_wind_speed_80m_degree_2',
 'mean_wind_speed_80m_degree_3',
 'mean_wind_speed_120m_degree_2',
 'mean_wind_speed_120m_degree_3',
 'mean_wind_speed_180m_degree_2',
 'mean_wind_speed_180m_degree_3',
 'mean_wind_speed_10m_meta_degree_2',
 'mean_wind_speed_10m_meta_degree_3',
 'mean_wind_speed_100m_degree_2',
 'mean_wind_speed_100m_degree_3',
# 'mean_wind_speed_80m_cut_degree_2',
# 'mean_wind_speed_80m_cut_degree_3',
 'mean_wind_speed_80m_curve_degree_2',
 'mean_wind_speed_80m_curve_degree_3',
 'mean_effect_wind_speed_10m_degree_2',
 'mean_effect_wind_speed_10m_degree_3',
 'mean_effect_wind_speed_80m_degree_2',
 'mean_effect_wind_speed_80m_degree_3',
 'mean_effect_wind_speed_120m_degree_2',
 'mean_effect_wind_speed_120m_degree_3',
 'mean_effect_wind_speed_180m_degree_2',
 'mean_effect_wind_speed_180m_degree_3',
 'mean_effect_wind_speed_10m_meta_degree_2',
 'mean_effect_wind_speed_10m_meta_degree_3',
 'mean_effect_wind_speed_100m_degree_2',
 'mean_effect_wind_speed_100m_degree_3',
 #'mean_effect_wind_speed_80m_cut_degree_2',
 #'mean_effect_wind_speed_80m_cut_degree_3',
 'mean_effect_wind_speed_80m_curve_degree_2',
 'mean_effect_wind_speed_80m_curve_degree_3',
 'kinetic_energy_80m',
 'kinetic_energy_80m_mean',
 'kinetic_energy_120m',
 'kinetic_energy_120m_mean',
 'kinetic_energy_80m_from_2m',
 'kinetic_energy_80m_mean_from_2m',
 'kinetic_energy_120m_from_2m',
 'kinetic_energy_120m_mean_from_2m',
 'sin_projection_energy_80m',
 'cos_projection_energy_80m',
 'sin_projection_energy_120m',
 'cos_projection_energy_120m',
 'toggle',
 'sin_hour',
 'cos_hour',
 'sin_month',
 'cos_month',
 'kinetic_energy_80m_lag_1',
 'kinetic_energy_80m_lag_2',
 'kinetic_energy_80m_lag_24',
 'kinetic_energy_120m_lag_1',
 'kinetic_energy_120m_lag_2',
 'kinetic_energy_120m_lag_24',
 'pressure_msl_lag_1',
 'pressure_msl_lag_2',
 'pressure_msl_lag_24',
 'temperature_80m_lag_1',
 'temperature_80m_lag_2',
 'temperature_80m_lag_24',
 'wind_speed_10m_meta_lag_1',
 'wind_speed_10m_meta_lag_2',
 'wind_speed_10m_meta_lag_24',
 'wind_direction_10m_meta_lag_1',
 'wind_direction_10m_meta_lag_2',
 'wind_direction_10m_meta_lag_24',
 'pressure_msl_meta_1_lag_2',
 'sqrt_diff_pressure_msl',
 'sqrt_diff_pressure_msl_meta']

hill_climb_from_random_07_meta = ['wind_speed_10m',
 'wind_speed_120m',
 'wind_speed_180m',
 'wind_direction_10m',
 'wind_direction_80m',
 'wind_direction_120m',
 'wind_direction_180m',
 'wind_gusts_10m',
 'temperature_80m',
 'temperature_120m',
 'pressure_msl',
 'rain',
 'showers',
 'snowfall',
 'cloud_cover_low',
 'count_repair',
 'wind_speed_10m_meta',
 'wind_direction_10m_meta',
 'temperature_2m',
 'pressure_msl_meta',
 'relative_humidity_2m',
 'cloud_cover_low_meta',
 'cloud_cover_mid',
 'cloud_cover_high',
 'rain_meta',
 'snowfall_meta',
 'precipitation',
# 'wind_direction_100m',
# 'wind_speed_100m',
 #'month_meta',
 #'hour_of_day_meta',
 'density_2m',
 'density_80m',
 'density_120m',
 'humidity',
 'sin_direction_10m',
 'sin_direction_80m',
 'cos_direction_80m',
 'sin_direction_120m',
 'cos_direction_120m',
 'sin_direction_180m',
 'cos_direction_180m',
 'sin_direction_meta',
 'cos_direction_meta',
# 'sin_direction_100m',
# 'cos_direction_100m',
 #'wind_speed_80m_cut',
 'wind_speed_80m_curve',
 'effect_wind_speed_10m',
 'effect_wind_speed_80m',
 'effect_wind_speed_120m',
 'effect_wind_speed_180m',
 'effect_wind_speed_10m_meta',
# 'effect_wind_speed_100m',
 #'effect_wind_speed_80m_cut',
 'effect_wind_speed_80m_curve',
 'mean_wind_speed_10m',
 'mean_wind_speed_80m',
 'mean_wind_speed_120m',
 'mean_wind_speed_180m',
 'mean_wind_speed_10m_meta',
# 'mean_wind_speed_100m',
 #'mean_wind_speed_80m_cut',
 'mean_wind_speed_80m_curve',
 'mean_effect_wind_speed_10m',
 'mean_effect_wind_speed_80m',
 'mean_effect_wind_speed_120m',
 'mean_effect_wind_speed_180m',
 'mean_effect_wind_speed_10m_meta',
# 'mean_effect_wind_speed_100m',
 #'mean_effect_wind_speed_80m_cut',
 'mean_effect_wind_speed_80m_curve',
 'wind_speed_10m_degree_2',
 'wind_speed_10m_degree_3',
 'wind_speed_80m_degree_2',
 'wind_speed_80m_degree_3',
 'wind_speed_120m_degree_2',
 'wind_speed_120m_degree_3',
 'wind_speed_180m_degree_2',
 'wind_speed_180m_degree_3',
 'wind_speed_10m_meta_degree_2',
 'wind_speed_10m_meta_degree_3',
# 'wind_speed_100m_degree_2',
# 'wind_speed_100m_degree_3',
 #'wind_speed_80m_cut_degree_2',
 #'wind_speed_80m_cut_degree_3',
 'wind_speed_80m_curve_degree_2',
 'wind_speed_80m_curve_degree_3',
 'effect_wind_speed_10m_degree_2',
 'effect_wind_speed_10m_degree_3',
 'effect_wind_speed_80m_degree_2',
 'effect_wind_speed_80m_degree_3',
 'effect_wind_speed_120m_degree_2',
 'effect_wind_speed_120m_degree_3',
 'effect_wind_speed_180m_degree_2',
 'effect_wind_speed_180m_degree_3',
 'effect_wind_speed_10m_meta_degree_2',
 'effect_wind_speed_10m_meta_degree_3',
 #'effect_wind_speed_100m_degree_2',
 #'effect_wind_speed_100m_degree_3',
 #'effect_wind_speed_80m_cut_degree_2',
 #'effect_wind_speed_80m_cut_degree_3',
 'effect_wind_speed_80m_curve_degree_2',
 'effect_wind_speed_80m_curve_degree_3',
 'mean_wind_speed_10m_degree_2',
 'mean_wind_speed_10m_degree_3',
 'mean_wind_speed_80m_degree_2',
 'mean_wind_speed_80m_degree_3',
 'mean_wind_speed_120m_degree_2',
 'mean_wind_speed_120m_degree_3',
 'mean_wind_speed_180m_degree_2',
 'mean_wind_speed_180m_degree_3',
 'mean_wind_speed_10m_meta_degree_2',
 'mean_wind_speed_10m_meta_degree_3',
 #'mean_wind_speed_100m_degree_2',
 #'mean_wind_speed_100m_degree_3',
 #'mean_wind_speed_80m_cut_degree_2',
 #'mean_wind_speed_80m_cut_degree_3',
 'mean_wind_speed_80m_curve_degree_2',
 'mean_wind_speed_80m_curve_degree_3',
 'mean_effect_wind_speed_10m_degree_2',
 'mean_effect_wind_speed_10m_degree_3',
 'mean_effect_wind_speed_80m_degree_2',
 'mean_effect_wind_speed_80m_degree_3',
 'mean_effect_wind_speed_120m_degree_2',
 'mean_effect_wind_speed_120m_degree_3',
 'mean_effect_wind_speed_180m_degree_2',
 'mean_effect_wind_speed_180m_degree_3',
 'mean_effect_wind_speed_10m_meta_degree_2',
 'mean_effect_wind_speed_10m_meta_degree_3',
 #'mean_effect_wind_speed_100m_degree_2',
 #'mean_effect_wind_speed_100m_degree_3',
 #'mean_effect_wind_speed_80m_cut_degree_2',
 #'mean_effect_wind_speed_80m_cut_degree_3',
 'mean_effect_wind_speed_80m_curve_degree_2',
 'mean_effect_wind_speed_80m_curve_degree_3',
 'kinetic_energy_80m',
 'kinetic_energy_80m_mean',
 'kinetic_energy_120m',
 'kinetic_energy_120m_mean',
 'kinetic_energy_80m_from_2m',
 'kinetic_energy_80m_mean_from_2m',
 'kinetic_energy_120m_from_2m',
 'kinetic_energy_120m_mean_from_2m',
 'sin_projection_energy_80m',
 'cos_projection_energy_80m',
 'sin_projection_energy_120m',
 'cos_projection_energy_120m',
 'toggle',
 'sin_hour',
 'cos_hour',
 'sin_month',
 'cos_month',
 'kinetic_energy_80m_lag_1',
 'kinetic_energy_80m_lag_2',
 'kinetic_energy_80m_lag_24',
 'kinetic_energy_120m_lag_1',
 'kinetic_energy_120m_lag_2',
 'kinetic_energy_120m_lag_24',
 'pressure_msl_lag_1',
 'pressure_msl_lag_2',
 'pressure_msl_lag_24',
 'temperature_80m_lag_1',
 'temperature_80m_lag_2',
 'temperature_80m_lag_24',
 'wind_speed_10m_meta_lag_1',
 'wind_speed_10m_meta_lag_2',
 'wind_speed_10m_meta_lag_24',
 'wind_direction_10m_meta_lag_1',
 'wind_direction_10m_meta_lag_2',
 'wind_direction_10m_meta_lag_24',
 'sqrt_diff_pressure_msl',
 'sqrt_diff_pressure_msl_meta']

lasso_01_features_plus_toggle = lasso_01_features + ["toggle"]

lasso_01_features_plus_toggle_and_seasonal_hour = lasso_01_features + ["sin_hour", "cos_hour", "toggle"]
lasso_01_features_plus_toggle_and_seasonal_hour = [col for col in lasso_01_features_plus_toggle_and_seasonal_hour if col != "hour_of_day"]

all_features_cols = [col for col in train.columns if col not in [
    "timestamp", TARGET_COL, "time_idx"
]]
all_features_cols_without_degree = [col for col in all_features_cols
                                if "degree" not in col] #Степени помогают

small_features_and_degree_cols

In [ ]:
sns.heatmap(train.corr())

In [ ]:
def time_split(percent_val=0.2):
    length_train = int(len(train) * (1 - percent_val))

    train_model = train.iloc[:length_train]
    val_model = train.iloc[length_train:]
    return train_model, val_model

def random_split(percent_val=0.2):
    x_train, x_val, y_train, y_val = train_test_split(
        train[[col for col in train.columns if col != TARGET_COL]],
         train[TARGET_COL],
         shuffle=True,
        random_state=SEED
    )
    x_train[TARGET_COL] = y_train
    x_val[TARGET_COL] = y_val
    return x_train, x_val

train.dropna(inplace=True)

#train["time_idx"] = np.arange(len(train))
train_model, val_model = time_split(percent_val=0.1)

if use_test:
    print(test)
    test.dropna(inplace=True) #Удаляются только строки, над которыми были произведены операции (последние не задеваются, в силу того, что power заполнены 0)

#Модели

##Метрика

In [ ]:
def metric(y_truth, predict):
    N = 90.09
    return np.mean(np.abs(y_truth - predict)) / N * 100

##Ансамблирование

###Параметры для моделей и их инициализация

In [ ]:
xgb_param_1 = {
        'objective': "reg:absoluteerror",
        'n_estimators': 973,
        'learning_rate': 0.0048847258097658944,
        'max_depth': 6,
        'subsample': 0.7234156898367853,
        'colsample_bytree': 0.9597847473447532,
        'lambda': 46.743764887013484,
        'alpha': 10.20593540492528906,
        "tree_method": "hist",
        "verbosity": 0,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_state": SEED
    }

xgb_param_2 = {
        'objective': "reg:absoluteerror",
        'n_estimators': 334,
        'learning_rate': 0.031321773330942354,
        'max_depth': 9,
        'subsample': 0.7427983732691806,
        'colsample_bytree': 0.8228809158389311,
        'lambda': 0.7941119527383735,
        'alpha': 94.57256459283009,
        "tree_method": "hist",
        "verbosity": 0,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_state": SEED
    }

xgb_param_3 = {
        'objective': "reg:absoluteerror",
        'n_estimators': 1281,
        'learning_rate': 0.010778249722610617,
        'max_depth': 4,
        'subsample': 0.6862046296540257,
        'colsample_bytree': 0.7179659293893328,
        'lambda': 253.06496753265077,
        'alpha': 27.28816676160678,
        "tree_method": "hist",
        "verbosity": 0,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_state": SEED
    }

xgb_param_4 = {
        'objective': "reg:absoluteerror",
        'n_estimators': 87,
        'learning_rate': 0.041917115166952006,
        'max_depth': 8,
        'subsample': 0.7020584494295802,
        'colsample_bytree': 0.7969909852161994,
        'lambda': 55.4709546679893,
        'alpha': 50.01622236700917195,
        "tree_method": "hist",
        "verbosity": 0,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_state": SEED
    }
xgb_param_5 = {
        'objective': "reg:quantileerror",
        "quantile_alpha": 0.5,
        'n_estimators': 442,
        'learning_rate': 0.011578586069502085,
        'max_depth': 4,
        'subsample': 0.718290468476387,
        'colsample_bytree': 0.8249707244513502,
        'lambda': 0.04766973482483789,
        'alpha': 0.09180952772726107,
        "tree_method": "hist",
        "verbosity": 0,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_state": SEED
    }

xgb_param_6 = {
        'objective': "reg:quantileerror",
        "quantile_alpha": 0.5,
        'n_estimators': 973,
        'learning_rate': 0.0048847258097658944,
        'max_depth': 7,
        'subsample': 0.7234156898367853,
        'colsample_bytree': 0.9597847473447532,
        'lambda': 151.93243956020044,
        'alpha': 100.03100303527029086,
        "tree_method": "hist",
        "verbosity": 0,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_state": SEED
    }

xgb_params = {
    "objective": "reg:absoluteerror",
    "n_estimators": 1000,
    "learning_rate": 0.005292881998626,
    "max_depth": 6,
    "subsample": 0.9417729089473964,
    "colsample_bytree": 0.9,
    "lambda": 0.372634183465811,
    "alpha": 0.832685112890902,
    "tree_method": "hist",
    "verbosity": 0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "random_state": SEED
}

lgb_param_1 = {
        'objective': "mae",
        'n_estimators': 677,
        'learning_rate': 0.006856019485669508,
        'num_leaves': 108,
        'subsample': 0.7242761179024555,
        'colsample_bytree': 0.7541373256008301,
        'reg_lambda': 100.1273474867160001,
        'reg_alpha': 100.026396761331930942,
        "verbose": -1,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_state": SEED
    }

lgb_param_2 = {
        'objective': "mae",
        'n_estimators': 385,
        'learning_rate': 0.01750292001385514,
        'num_leaves': 57,
        'subsample': 0.7000581952680016,
        'colsample_bytree': 0.7022407760313027,
        'reg_lambda': 10.6741849320181039,
        'reg_alpha': 10.020069572919248886,
        "verbose": -1,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_state": SEED
    }

lgb_param_3 = {
        "objective":  "quantile",
        "alpha": 0.5,
        'n_estimators': 222,
        'learning_rate': 0.026079656598095833,
        'num_leaves': 256,
        'subsample': 0.7835558684167139,
        'colsample_bytree': 0.7418481581956126,
        'reg_lambda': 100.23592770353071293,
        'reg_alpha': 100.526651484110799,
        "verbose": -1,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_state": SEED
    }
lgb_param_4 = {
        "objective":  "quantile",
        "alpha": 0.5,
        'n_estimators': 877,
        'learning_rate': 0.006868795465704379,
        'num_leaves': 101,
        'subsample': 0.624507413329952,
        'colsample_bytree': 0.68963064294269,
        'reg_lambda': 40.088989738160468,
        'reg_alpha': 50.01544365106217038,
        "verbose": -1,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_state": SEED
    }

lgb_params = {
        "objective": "quantile", #"mae"
        "alpha": 0.5,
        "n_estimators": 250,
        "learning_rate": 0.025468037883334,
        "num_leaves": 68,
        "subsample": 0.8614381770563153,
        "colsample_bytree": 0.8411016230697343,
        "reg_lambda": 15.13164984536291765,
        "reg_alpha": 12.664909586669296,
        "verbose": -1,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_state": SEED
}

cat_param_1 = {
        "loss_function": "MAE",
        'n_estimators': 877,
        'learning_rate': 0.048113056573625794,
        'max_depth': 5,
        'subsample': 0.7932946965146986,
        'l2_leaf_reg': 26.81621920201293,
        "verbose": False,
        "random_state": SEED
        }
cat_param_2 = {
        "loss_function": "MAE",
        'n_estimators': 381,
        'learning_rate': 0.07302183721522207,
        'max_depth': 6,
        'subsample': 0.9187025677641707,
        'l2_leaf_reg': 0.22139022748632253,
        "verbose": False,
        "random_state": SEED
    }
cat_param_3 = {
        "loss_function": "Quantile",
        'n_estimators': 402,
        'learning_rate': 0.10436841187109172,
        'max_depth': 5,
        'subsample': 0.8590732232023351,
        'l2_leaf_reg': 15.3476731914222015,
        "verbose": False,
        "random_state": SEED
    }
cat_param_4 = {
         "loss_function": "Quantile",
         'n_estimators': 877,
         'learning_rate': 0.048113056573625794,
         'max_depth': 5,
         'subsample': 0.7932946965146986,
         'l2_leaf_reg': 9.905204218383902,
         "verbose": False,
         "random_state": SEED
     }

cat_params = {
        "loss_function": "MAE",
        "n_estimators": 750,
        "learning_rate": 0.025468037883334,
        "max_depth": 8,
        "l2_leaf_reg": 0.09894212515987615,
        "bagging_temperature": 1.9576501770171428,
        "verbose": False,
       # "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_state": SEED
}

ridge_param = {
    "alpha": 5.0,
    "random_state": SEED
}

rf_param = {
    "n_estimators": 100,
    "max_depth": 4,
    "random_state": SEED,
    "criterion": "absolute_error"
}

knn_params = {
    "n_neighbors": 5,
    "weights": "distance"
}

In [ ]:
use_configuration = hill_climb_from_random_meta
categorical_features = []#["toggle", "count_repair"]
numeric_features = [col for col in use_configuration if col not in categorical_features]

ohe = OneHotEncoder()
transform_train = train.drop(columns=categorical_features).copy()

ohe_data = ohe.fit_transform(train[categorical_features]).toarray()
ohe_frame = pd.DataFrame(ohe_data, columns=ohe.get_feature_names_out(), index=transform_train.index)

transform_train = pd.concat([transform_train, ohe_frame], axis=1)

use_configuration = numeric_features + list(ohe_frame.columns)

transform_valid = valid.drop(columns=categorical_features).copy()

ohe_data = ohe.transform(valid[categorical_features]).toarray()
ohe_frame = pd.DataFrame(ohe_data, columns=ohe.get_feature_names_out(), index=transform_valid.index)

transform_valid = pd.concat([transform_valid, ohe_frame], axis=1)
transform_valid[use_configuration].head()

transform_test = test.drop(columns=categorical_features).copy()

ohe_data = ohe.transform(test[categorical_features]).toarray()
ohe_frame = pd.DataFrame(ohe_data, columns=ohe.get_feature_names_out(), index=transform_test.index)

transform_test = pd.concat([transform_test, ohe_frame], axis=1)
transform_test[use_configuration].head()

#transform_train = pd.concat([transform_train, transform_test.iloc[:-24]], axis=0)
transform_test = transform_test.iloc[-24:]
transform_test.head()

### Ансамбль учится на всех данных

In [ ]:
from copy import deepcopy
import joblib


def all_ensemble(models, data, data_predict, target_name=TARGET_COL, n_folds=10, SEEDs=None, use_time_split=True, test_size=len(valid)):
    if SEEDs is None:
        SEEDs = [SEED]
    feature_cols = [col for col in data.columns if col != target_name]
    predicts = []
    models_all = []
    for seed_number, seed in enumerate(SEEDs):
        print(f"seed = {seed}")
        for i in range(len(models)):
            print(f"Model: {i + 1}")
            model_copy = deepcopy(models[i])
            if i != 0:
                model_copy.set_params(random_state=seed)
            model_copy.fit(data[feature_cols], data[target_name])
            predicts.append(model_copy.predict(data_predict))
            models_all.append(model_copy)
            joblib.dump(model_copy, f'model_{i + 1}.pkl')
    return predicts, models_all

In [ ]:
predicts_copy, models_all = all_ensemble(models=(
    xgb.XGBRegressor(**xgb_param_1),
    xgb.XGBRegressor(**xgb_param_2),
    xgb.XGBRegressor(**xgb_param_3),
    xgb.XGBRegressor(**xgb_param_4),
    xgb.XGBRegressor(**xgb_param_5),
    xgb.XGBRegressor(**xgb_param_6),
    xgb.XGBRegressor(**xgb_params),
    lgb.LGBMRegressor(**lgb_param_1),
    lgb.LGBMRegressor(**lgb_param_2),
    lgb.LGBMRegressor(**lgb_param_3),
    lgb.LGBMRegressor(**lgb_param_4),
    lgb.LGBMRegressor(**lgb_params),
    CatBoostRegressor(**cat_param_1),
    CatBoostRegressor(**cat_param_2),
    CatBoostRegressor(**cat_param_3),
    CatBoostRegressor(**cat_param_4),
    CatBoostRegressor(**cat_params),
    Ridge(**ridge_param),
    Lasso(**ridge_param)
    #RandomForestRegressor(**rf_param)
), data=transform_train[use_configuration + [TARGET_COL]],
    data_predict=transform_valid[use_configuration],
                                     use_time_split=False, test_size=len(valid),
                                         SEEDs=[SEED])

In [ ]:
np.array(predicts_copy).shape

In [ ]:
predicts = np.array(predicts_copy)
predicts = np.average(predicts, axis=0, weights=None)
predicts = np.clip(predicts, a_min=0, a_max=88)
predicts = predicts[::-1]

In [ ]:
predict_df = pd.DataFrame({"predict": predicts})
save_path = "/content/drive/MyDrive/ML_data/wind_farm/submissions/ensemble_meta.csv"

predict_df.to_csv(save_path, index=False)

##Инференс (запускается строго после какой-то из ячеек модели)

In [ ]:
predict = model.predict(valid[use_configuration])
predict = predict[::-1] #так как в начале сортировали
plt.plot(predict)

In [ ]:
predict_df = pd.DataFrame({"predict": predict})
save_path = "/content/drive/MyDrive/ML_data/wind_farm/submissions/cat_baseline.csv"

predict_df.to_csv(save_path, index=False)